# 02 - Shared Preprocessing Setup

Milestone 2: Creating the common data contract and validation split used by the text and image preprocessing workflows.


## 1. Purpose

The Rakuten challenge provides labels for the training catalogue but not for `X_test_update.csv`. Local model evaluation therefore requires a held-out part of the labelled training data.

This notebook is the single owner of the shared training/validation assignment. It:

- loads and validates the labelled training catalogue;
- distinguishes the official test catalogue from local validation data;
- creates or reuses a stratified 80/20 split;
- stores the assignment under `data/processed/data_split.csv`;
- verifies coverage and class balance;
- defines the split contract consumed by the text and image notebooks.

Modality-specific cleaning and transformations do not belong in this notebook.


## 2. Imports and Configuration


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

from src.data_utils import get_data_paths, load_merged_data

RANDOM_STATE = 42
VALIDATION_SIZE = 0.20

pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Environment and shared data utilities loaded successfully.")


## 3. Load and Validate the Challenge Tables

The original dataframe index is preserved as `row_id`. It is the common key used to join split assignments, text features, image metadata and model predictions.


In [ ]:
data_paths = get_data_paths()

train_df = (
    load_merged_data()
    .rename_axis("row_id")
    .reset_index()
)
test_df = (
    pd.read_csv(data_paths["x_test"], index_col=0)
    .rename_axis("row_id")
    .reset_index()
)

assert train_df["row_id"].is_unique
assert test_df["row_id"].is_unique
assert train_df["prdtypecode"].notna().all()
assert train_df["prdtypecode"].nunique() == 27
assert "prdtypecode" not in test_df.columns

dataset_summary = pd.DataFrame({
    "source": ["labelled training catalogue", "official unlabelled test catalogue"],
    "rows": [len(train_df), len(test_df)],
    "target_available": [True, False],
})
display(dataset_summary)


## 4. Dataset Roles

The official test catalogue cannot be used to calculate F1 scores because its labels are hidden. It is reserved for final blind predictions.

The local evaluation sets are therefore created only from the labelled catalogue:

```text
Labelled catalogue
├── train       80%
└── validation  20%

Official test catalogue
└── final predictions only
```


## 5. Create or Load the Shared Split

The baseline split is stratified by `prdtypecode`, uses `random_state=42`, and assigns 20% of the labelled products to validation. Existing assignments are reused so repeated notebook runs cannot silently reshuffle the evaluation set.

This first baseline is stratified but not duplicate-aware. The audit found repeated titles and exact duplicate images, so a grouped split should later be evaluated as a leakage-sensitivity check. It should not silently replace the baseline because both validation strategies answer useful but different questions.


In [ ]:
split_path = (
    Path(data_paths["x_train"]).parent.parent
    / "processed"
    / "data_split.csv"
)

if split_path.exists():
    split_manifest = pd.read_csv(split_path, usecols=["row_id", "split"])
    split_action = "Loaded existing"
else:
    _, validation_rows = train_test_split(
        train_df[["row_id", "prdtypecode"]],
        test_size=VALIDATION_SIZE,
        random_state=RANDOM_STATE,
        stratify=train_df["prdtypecode"],
    )

    split_manifest = train_df[["row_id"]].copy()
    split_manifest["split"] = "train"
    split_manifest.loc[
        split_manifest["row_id"].isin(validation_rows["row_id"]),
        "split",
    ] = "validation"

    split_path.parent.mkdir(parents=True, exist_ok=True)
    split_manifest.to_csv(split_path, index=False)
    split_action = "Created new"

print(f"{split_action} split manifest: {split_path}")


## 6. Validate Coverage and Class Balance

A stale, incomplete or ambiguous split file must fail before it reaches either modality-specific workflow.


In [ ]:
assert split_manifest["row_id"].is_unique
assert split_manifest["split"].notna().all()
assert set(split_manifest["split"]) == {"train", "validation"}
assert set(split_manifest["row_id"]) == set(train_df["row_id"])

prepared_df = train_df.merge(
    split_manifest,
    on="row_id",
    how="left",
    validate="one_to_one",
)
assert prepared_df["split"].notna().all()

split_summary = prepared_df.groupby("split").agg(
    products=("row_id", "size"),
    classes=("prdtypecode", "nunique"),
)
split_summary["share"] = split_summary["products"] / len(prepared_df)

validation_share = split_summary.loc["validation", "share"]
assert abs(validation_share - VALIDATION_SIZE) <= (1 / len(prepared_df))

class_shares = pd.crosstab(
    prepared_df["prdtypecode"],
    prepared_df["split"],
    normalize="columns",
)
class_shares["absolute_difference"] = (
    class_shares["train"] - class_shares["validation"]
).abs()
max_class_share_difference = class_shares["absolute_difference"].max()

display(split_summary.style.format({"share": "{:.1%}"}))
print(
    "Maximum absolute class-share difference between train and validation: "
    f"{max_class_share_difference:.3%}"
)
display(class_shares.sort_values("absolute_difference", ascending=False).head(10).style.format("{:.3%}"))


## 7. Visual Class-Balance Check

The chart compares class shares rather than raw counts because the training and validation partitions have different sizes. Closely aligned bars confirm that stratification preserved the target distribution.


In [ ]:
plot_data = (
    class_shares[["train", "validation"]]
    .reset_index()
    .melt(
        id_vars="prdtypecode",
        var_name="split",
        value_name="class_share",
    )
)

fig, axis = plt.subplots(figsize=(14, 6))
for split_name, colour, offset in [
    ("train", "#BF0000", -0.2),
    ("validation", "#64748B", 0.2),
]:
    subset = plot_data.loc[plot_data["split"].eq(split_name)]
    positions = range(len(subset))
    axis.bar(
        [position + offset for position in positions],
        subset["class_share"],
        width=0.4,
        label=split_name.title(),
        color=colour,
    )

class_codes = class_shares.index.astype(str).tolist()
axis.set_xticks(range(len(class_codes)))
axis.set_xticklabels(class_codes, rotation=60, ha="right")
axis.set_xlabel("Product type code")
axis.set_ylabel("Share within split")
axis.set_title("Class distribution in the shared training and validation split")
axis.legend()
plt.tight_layout()
plt.show()


## 8. Contract for Modality-Specific Notebooks

Both `02_preprocessing_image.ipynb` and the future `02_preprocessing_text.ipynb` must:

1. load `data/processed/data_split.csv`;
2. join it to the labelled catalogue through `row_id`;
3. validate the join as one-to-one;
4. train preprocessing or modelling components only on rows labelled `train`;
5. reserve rows labelled `validation` for evaluation;
6. keep the official test catalogue separate and unlabelled.

Because the entire `data/` directory is excluded from Git, team members who do not receive the manifest separately must run this notebook before running either modality-specific notebook.


## 9. Result and Next Step

The shared baseline split is now defined independently of text and image preprocessing. The modality notebooks can be developed in parallel while evaluating both classifiers on exactly the same held-out products.

The next steps are to implement text cleaning in `02_preprocessing_text.ipynb` and image transformations in `02_preprocessing_image.ipynb`. Model fitting remains part of the modelling milestone.
